In [ ]:
import pandas as pd
import numpy as np
from ChromaVDB.chroma import ChromaFramework
from DeepGraphDB import DeepGraphDB
from tqdm.notebook import tqdm
import torch
import pickle

gdb = DeepGraphDB()
gdb.load_graph("/home/cc/PHD/dglframework/DeepKG/DeepGraphDB/graphs/primekg.bin")

vdb = ChromaFramework(persist_directory="./ChromaVDB/chroma_db")
records = vdb.list_records()

gene_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'graph' and record['entity'] == 'geneprotein']
gene_names = [record['name'] for record in records if record['embedding_type'] == 'graph' and record['entity'] == 'geneprotein']

target = 35884

subg = gdb.get_k_hop_neighbors([target], k=2)

flat_nodes = []

for key, value in subg.items():
    flat_nodes.extend(value)

flat_nodes = list(set(flat_nodes)) 

gene_subg_names = [ gdb.node_data['geneprotein']['name'][gdb.global_to_local_mapping[fi][1]] for fi in flat_nodes \
                    if gdb.global_to_local_mapping[fi][0] == 'geneprotein' ] 

gene_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'graph' and record['entity'] == 'geneprotein' \
             and record['name'] in gene_subg_names]

gene_names = [record['name'] for record in records if record['embedding_type'] == 'graph' and record['entity'] == 'geneprotein' \
              and record['name'] in gene_subg_names]

In [ ]:
def process_gene_list(gene_data: list[dict], ordered_names: list[str]) -> tuple[list[int], list[str], list[int]]:
    """
    Filters a list of gene names and returns their classes and original indices.

    Args:
        gene_data: A list of dictionaries, e.g., [{'name': 'NRF1', 'gene_class': 0}, ...].
        ordered_names: A list of gene names specifying the desired order.

    Returns:
        A tuple containing three lists:
        - The gene classes for the valid names.
        - The list of valid gene names.
        - The original indices of the valid names from the input `ordered_names` list.
    """
    # 1. Create a mapping of name -> gene_class for efficient lookups.
    gene_class_map = {item['name']: item['gene_class'] for item in gene_data}
    
    # 2. Initialize lists to store the results.
    final_classes = []
    valid_names = []
    original_indices = []
    
    # 3. Iterate through the names list with its index.
    for index, name in enumerate(ordered_names):
        # Check if the name exists in our map.
        if name in gene_class_map:
            # If it exists, add its info to our results lists.
            valid_names.append(name)
            original_indices.append(index)
            final_classes.append(gene_class_map[name])
            
    return final_classes, valid_names, original_indices

with open("/home/cc/PHD/dglframework/DeepKG/gene_classes.pkl", "rb") as f:
    gene_labels = pickle.load(f)


In [ ]:
classes, valid_names, original_indices = process_gene_list(gene_labels, gene_names)
# TODO: sarebbe interessante vedere date come si distribuiscono i geni rispetto agli embs del fine tuning. Bisogna capire se "geneproteint.pt" è ordinato rispetto al local indexing

gene_embs = [gene_embs[i] for i in original_indices]

In [ ]:
gene_embs_ft = torch.load("/home/cc/PHD/dglframework/DeepKG/finetune-embs/geneprotein.pt")
gene_embs = gene_embs_ft

with open("/home/cc/PHD/dglframework/DeepKG/conditioning.pkl", "rb") as f:
    conditioning = pickle.load(f)

nodes_to_keep = {}
gene_names_ft = []

conditioning.append({ 'entity': 'disease: diffuse large B-cell lymphoma', 'score': 5.0 })

for item in conditioning:
    entity = item['entity'].split(': ')
    score = item['score']

    if entity[1] in gdb.node_data[entity[0]]['name']:
        idx = np.where(gdb.node_data[entity[0]]['name'] == entity[1])[0][0]

        nodes_to_keep[entity[0]] = nodes_to_keep.get(entity[0], []) + [idx]
        if entity[0] == 'geneprotein':
            gene_names_ft.append(entity[1])

classes, valid_names, original_indices = process_gene_list(gene_labels, gene_names_ft)

In [ ]:
import json
from typing import List, Dict, Tuple, Any

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.pydantic_v1 import BaseModel, Field

# Define the output schema for LangChain's PydanticOutputParser
class ScoredEntity(BaseModel):
    """An entity from the biological knowledge graph with an assigned importance score."""
    name: str = Field(description="The name of the gene (e.g. TP53)")
    gene_class: int = Field(description="The number of the class from 0 to 3, relative to the gene classification.")

class EntityScoreOutput(BaseModel):
    """List of important entities and their scores."""
    important_entities: List[ScoredEntity] = Field(
        description="A list of genes with associated class"
    )

# Helper function to escape curly braces in a string
def escape_curly_braces(text: str) -> str:
    """Escapes single curly braces to double curly braces for f-string compatibility."""
    # Replace { with {{ and } with }}
    return text.replace("{", "{{").replace("}", "}}")

def gene_classification(genes_list: List[str], ollama_model_name: str = "alibayram/medgemma:27b") -> List[Dict[str, Any]]:

    # Initialize the Ollama LLM with structured output directly
    llm = ChatOllama(model=ollama_model_name).with_structured_output(EntityScoreOutput)
    
    # Define the prompt template
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful assistant specialized in classifying genes based on their functional roles in the human body. "
                "For each gene, classify it according to the four functional categories defined below. Using the number of the class (0 to 3) to indicate the class of the gene. "
                "A single gene can belong to multiple categories if its functions apply. Provide the classification for each gene.\n"
                "1.  **Transcriptional & Epigenetic Regulators (class 0):** Identify genes whose products directly control gene expression."
                "includes proteins that bind to DNA to activate or repress transcription (transcription factors, "
                "co-activators, repressors) and enzymes that chemically modify histones or DNA to alter "
                "chromatin structure and accessibility (e.g., histone methyltransferases, acetyltransferases, "
                "demethylases). Also include core structural chromatin proteins like histones.\n"
                "Key Functions to Look For: Transcription factor activity, DNA binding, histone modification, "
                "chromatin remodeling, gene silencing, transcriptional activation/repression.\n"
                "2.  **Cell Cycle & Apoptosis Regulators (class 1):** Identify genes whose products function as critical control points in cell "
                "division or programmed cell death (apoptosis). This includes proteins that enforce cell cycle"
                "checkpoints (e.g., G1/S, G2/M), promote or inhibit proliferation (oncogenes, tumor "
                "suppressors), or are essential components of the apoptotic signaling cascade (e.g., death "
                "receptors, caspases, mitochondrial pathway members).\n"
                "Key Functions to Look For: Cell cycle arrest, apoptosis, programmed cell death, proliferation, tumor suppression, checkpoint control.\n"
                "3.  **Cell Signaling & Communication (class 2):** Identify genes whose products act as core components of signal"
                "transduction pathways. This includes cell surface receptors that bind to external ligands, "
                "intracellular kinases and phosphatases that relay signals through phosphorylation, and "
                "enzymes that generate, modify, or degrade signaling molecules to"
                "modulate information flow within or between cells.\n"
                "Key Functions to Look For: Signal transduction, kinase activity, receptor activity, G-protein signaling, NF-κB pathway, MAPK pathway, ubiquitin-editing."
                "4.  **Immune Development & Response (class 3):** Identify genes with a specialized role in the development, maturation, or "
                "function of the immune system. This includes genes that regulate the differentiation of "
                "immune cells (e.g., B-cells, T-cells), mediate the inflammatory response, control immune"
                "tolerance, or are directly involved in recognizing and eliminating pathogens or malignant cells.\n"
                "Key Functions to Look For: Lymphocyte differentiation, immune response, inflammation, cytokine signaling, T-cell/B-cell activation, negative regulation of immunity.\n"
                "**Output Format:**\n"
                "Provide your response as a JSON object, specifically as a list under the key 'important_entities'. "
                "Each item in the list should be a dictionary containing 'name' (the name of the gene)"
                "and its assigned 'gene_class'. Provide a class for ALL the genes provided in 'List of genes' (reply with a class from 0 to 3).\n"
            ),
            (
                "human",
                "**List of genes:**\n{genes_list}\n"
                "**Begin your classification.**"
            ),
        ]
    )

    # Create the LangChain chain
    chain = prompt | llm

    # Invoke the chain with the subgraph data
    response = chain.invoke({
        "genes_list": genes_list,
    })
    
    return response.important_entities

In [ ]:
# batch_size = 10

# genes = gene_names
# genes_class = []

# for i in tqdm(range(0, len(genes), batch_size)):
#     batch_genes = genes[i:i + batch_size]
    
#     scored_entities = gene_classification(batch_genes, ollama_model_name="alibayram/medgemma:27b")
    
#     genes_class.extend(scored_entities)

# genes_class = [ gene.dict()['gene_class'] for gene in genes_class ]

# with open("/home/cc/PHD/dglframework/DeepKG/gene_classes.pkl", "wb") as fp: 
#     pickle.dump(genes_class, fp)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import torch


def visualize_embeddings_tsne(embeddings, labels, perplexity=30, n_iter=1000, random_state=42):
    """
    Visualize embeddings using t-SNE for multiple classes. 🎨

    Args:
        embeddings: A list of tensors/numpy arrays, or a single tensor/numpy array.
        labels: A list or array of integer labels for each embedding.
        perplexity (int): t-SNE perplexity parameter (default: 30).
        n_iter (int): Number of iterations for t-SNE (default: 1000).
        random_state (int): Random state for reproducibility (default: 42).
    """
    
    ## 1. Data Preparation
    # Convert various input types into a single NumPy array
    if isinstance(embeddings, list):
        if torch.is_tensor(embeddings[0]):
            embs_np = torch.stack(embeddings).detach().cpu().numpy()
        else:
            embs_np = np.array(embeddings)
    elif torch.is_tensor(embeddings):
        embs_np = embeddings.detach().cpu().numpy()
    else:
        embs_np = np.array(embeddings)
    
    # Ensure the embeddings array is 2D
    if embs_np.ndim > 2:
        embs_np = embs_np.reshape(embs_np.shape[0], -1)
    
    labels_np = np.array(labels)
    unique_labels = np.unique(labels_np)
    n_classes = len(unique_labels)
    
    print(f"Embedding shape: {embs_np.shape}")
    print(f"Found {n_classes} unique classes: {unique_labels}")
    
    ## 2. t-SNE Computation
    print("Applying t-SNE...")
    tsne = TSNE(n_components=2, init='pca', perplexity=perplexity, n_iter=n_iter, random_state=random_state)
    embeddings_2d = tsne.fit_transform(embs_np)
    
    ## 3. Plotting
    plt.figure(figsize=(12, 10))
    
    # Get a color map to generate distinct colors for each class
    cmap = plt.cm.get_cmap('viridis', n_classes)
    
    # Plot points for each class with a different color
    for i, label in enumerate(unique_labels):
        mask = (labels_np == label)
        plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                    color=cmap(i), label=f'Class {label}', 
                    alpha=0.8, s=50)
    
    plt.title('t-SNE Visualization of Embeddings', fontsize=16)
    plt.xlabel('t-SNE Component 1', fontsize=12)
    plt.ylabel('t-SNE Component 2', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
    
    return embeddings_2d, tsne

visualize_embeddings_tsne(gene_embs, classes, perplexity=55)